# 06 - Backward 7-day mean of the clustering input

Reads the latents saved by notebooks 02 and 03 and clusters them again with the backward 7-day mean: of the latents and the price level for the daily models, and of the price level only for the rolling weekly models. No model is trained.

Output: `experiments/vae_lstm/latent_smoothing/regimes.csv` (one column per input) and `inputs.csv`.

<d.benavidess@uniandes.edu.co>

In [ ]:
import sys
sys.path.append("..")

import json
from pathlib import Path

import numpy as np
import pandas as pd

from src.data import load_context, date_split
from src.clustering import level_columns, fit_clusters, order_regimes
from src import experiments as exp

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## 0. Configuration

`studies` are the VAE-LSTM studies whose trained models are read; their `artifacts/latents.npy` must be on this computer.
- `k_values` is always 3, 4 and 5.

In [ ]:
scalogram_sets = ["real_price_log", "real_price_log_weekly"]
studies = ["vae_lstm/daily", "vae_lstm/weekly"]
daily_set = "real_price_log"
weekly_set = "real_price_log_weekly"
k_values = [3, 4, 5]
window = 7
out_dir = exp.EXPERIMENTS_DIR / "vae_lstm" / "latent_smoothing"

## 1. Data and trained models

In [ ]:
sets = {}
for scalogram_set in scalogram_sets:
    metadata = pd.read_csv(Path("../data/processed/scalograms") / scalogram_set / "scalogram_metadata.csv",
                           parse_dates=["start_datetime", "end_datetime"])
    context = load_context(Path("../data/processed"), metadata)
    sets[scalogram_set] = {"metadata": metadata, "dates": context["date"].dt.date,
                           "train_idx": date_split(context["date"])[0],
                           "price": context["precio_real_cop_kwh"].to_numpy()}

models = {}
for study in studies:
    for path in sorted((exp.EXPERIMENTS_DIR / study).glob("*/config.json")):
        cfg = json.loads(path.read_text())
        if cfg.get("scalogram_set") in sets and cfg.get("trained_in"):
            models.setdefault(cfg["trained_in"], {"study": cfg["study"], "scalogram_set": cfg["scalogram_set"],
                                                  "z_dim": cfg["z_dim"], "lam": cfg["lam"], "beta": cfg["beta"],
                                                  "seed": cfg["seed"]})
models = pd.DataFrame.from_dict(models, orient="index").rename_axis("run_id").reset_index()

## 2. Clustering

The price columns are z-scored with the training days; the latents are used as they are.

In [ ]:
def zscore(x, train_idx):
    x = np.asarray(x, dtype=np.float64)
    return (x - x[train_idx].mean(axis=0)) / x[train_idx].std(axis=0)


def backward_mean(x):
    return pd.DataFrame(x).rolling(window, min_periods=1).mean().to_numpy()


columns, inputs = {}, []
for scalogram_set, s in sets.items():
    set_models = models[models["scalogram_set"] == scalogram_set] if len(models) else models
    if set_models.empty:
        continue
    train_idx = s["train_idx"]
    level = s["metadata"][list(level_columns(s["metadata"]))].to_numpy(dtype=np.float64)
    level_7d = zscore(backward_mean(level), train_idx)
    features = "latents + level 7d" if scalogram_set == weekly_set else "latents 7d + level 7d"
    for _, model in set_models.iterrows():
        latents = np.load(exp.EXPERIMENTS_DIR / model["run_id"] / "artifacts" / "latents.npy").astype(np.float64)
        # the weekly latents already cover 7 days, so only the daily ones get the backward 7-day mean
        x = np.hstack([latents if scalogram_set == weekly_set else backward_mean(latents), level_7d])
        for k in k_values:
            name = f"c{len(inputs):03d}"
            regimes = order_regimes(fit_clusters(x, train_idx, k, int(model["seed"]))[0], s["price"])
            columns[name] = pd.Series(regimes, index=s["dates"], dtype="Int64")
            inputs.append({"column": name, "model": "vae_lstm", **model.drop("scalogram_set").to_dict(), "k": k,
                           "features": features, "smoothed": True})

out_dir.mkdir(parents=True, exist_ok=True)
pd.DataFrame(columns).sort_index().rename_axis("date").reset_index().to_csv(out_dir / "regimes.csv", index=False)
pd.DataFrame(inputs).to_csv(out_dir / "inputs.csv", index=False)
print(f"{len(models)} models, {len(inputs)} regime columns")